# gru_va board session #1 — expansion music + battery, cells 1–2 (REV1)
Sentinel: **BOARDS1-REV1-2026-08-14**

Scope: `rodent_mod` + `gt_max` × 17 widths. Per config: golden gate
(**byte-compare REQUIRED** vs staged B1-certified `ref_gold_*`), 4 music
sources (`board_<src>_<tag>.f32`), 5-stimulus battery (`s2_<class>_<tag>.f32`,
fleet-battery scope addition 2026-08-14). No throughput block (fleet-complete).

Runbook:
1. Run staging (`stage_session1_REV1.ps1` on Lenny) BEFORE opening this.
2. Cells top to bottom. Helpers cell hard-verifies every stimulus MD5
   (battery + golden vs embedded certs; music vs `stim_manifest_session1.csv`).
3. J1 preflight must show 0 missing bitstreams / refs before J2.
4. J2 is resumable (provenance-aware done-set keyed on stim MD5).
5. Disk: ~5.4 GB written per cell. J2 hard-stops below 400 MB free —
   do the mid-session pull+verify on Lenny, delete pulled outputs via C1,
   re-run J2 (it resumes).
6. RTNeural ARM baseline row: separate PS-side task at session close
   (schedule fallback Mon 8/17) — not part of this notebook.


## 1. Configuration

In [ ]:
import csv, hashlib, os, time
import numpy as np
from pynq import Overlay, allocate
from pynq.ps import Clocks

SENTINEL    = "BOARDS1-REV1-2026-08-14"
SESSION_DIR = "/home/xilinx/jupyter_notebooks/session1"
MANIFEST    = os.path.join(SESSION_DIR, "session1_manifest.csv")
STIM_MAN    = os.path.join(SESSION_DIR, "stim_manifest_session1.csv")
RESULTS     = os.path.join(SESSION_DIR, "session1_results.csv")
GOLDEN_IN   = os.path.join(SESSION_DIR, "golden_input.bin")
IP_NAME     = "gru_va_0"
DMA_NAME    = "axi_dma_0"
CHUNK       = 65536
FCLK_MHZ    = 100.0
GOLDEN_N    = 4096
STIM_CELLS = ["rodent_max"]      # session 4 = anchor symmetry capture

MUSIC_STIMS = {                              # run order: music first
    "gtr2":    "anchor_gtr2_in.f32",
    "gtr4sg":  "anchor_gtr4sg_in.f32",
    "prvtgtr": "anchor_prvtgtr_in.f32",
    "ytbass":  "anchor_ytbass_in.f32",
}
BATT_STIMS = {                               # then battery (scope addition 8/14)
    "tones":   "stim_tones_in.f32",
    "sweep":   "stim_sweep_in.f32",
    "silence": "stim_silence_in.f32",
    "impulse": "stim_impulse_in.f32",
    "edge":    "stim_edge_in.f32",
}
STIMS = dict(list(MUSIC_STIMS.items()) + list(BATT_STIMS.items()))

# STIMGEN-REV2 birth certificates + golden cert (embedded, hard gate)
EXPECTED_MD5 = {
    "stim_tones_in.f32":   "3991acd46657e9051419bb3fbc24ff74",
    "stim_sweep_in.f32":   "913d44e192cffd1459eb55e56bdd6be1",
    "stim_silence_in.f32": "d051cff927d4ce28ab72215e43a2fb2b",
    "stim_impulse_in.f32": "eb445918179de211aa2625a4480c4be3",
    "stim_edge_in.f32":    "fad9029aef784288d56fff14cb2f9fc3",
    "golden_input.bin":    "607f8f6fbf8d191b8b35c264a665a829",
}

print(SENTINEL)
st = os.statvfs(SESSION_DIR)
print("free space: %.1f GB (session writes ~10.8 GB total; ~5.4 GB per cell)"
      % (st.f_bavail*st.f_frsize/1e9))


## 2. Helpers + staged-file provenance (hard MD5 gates; run after any restaging)

In [ ]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""):
            h.update(ch)
    return h.hexdigest()

def md5_of(arr):
    return hashlib.md5(arr.tobytes()).hexdigest().upper()

def start_kernel(ip):
    ip.register_map.CTRL.AP_START = 1

def wait_done(ip, timeout_s=15.0):
    t0 = time.time()
    while int(ip.register_map.CTRL.AP_IDLE) != 1:
        if time.time() - t0 > timeout_s:
            raise TimeoutError("kernel not idle after %.1fs" % timeout_s)

def load_bitstream(bit_name):
    """Overlay load + clock fix + handle fetch, as one inseparable action."""
    ol = Overlay(os.path.join(SESSION_DIR, bit_name))
    Clocks.fclk0_mhz = FCLK_MHZ
    assert abs(Clocks.fclk0_mhz - FCLK_MHZ) < 0.5, "fclk0 set failed: %s" % Clocks.fclk0_mhz
    ip  = getattr(ol, IP_NAME)
    dma = getattr(ol, DMA_NAME)
    return ol, ip, dma

def run_golden(ip, dma, golden_x, ref):
    """4096-sample gate vs B1-certified ref. Byte-compare REQUIRED."""
    frag = {}
    ibuf = allocate(shape=(GOLDEN_N,), dtype=np.float32)
    obuf = allocate(shape=(GOLDEN_N,), dtype=np.float32)
    try:
        ibuf[:] = golden_x; ibuf.flush()
        rm = ip.register_map
        rm.mode = 1; rm.n_samples = GOLDEN_N; rm.reset_state = 1
        dma.recvchannel.transfer(obuf); start_kernel(ip); dma.sendchannel.transfer(ibuf)
        dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
        obuf.invalidate()
        bg = np.asarray(obuf).copy()
    finally:
        ibuf.freebuffer(); obuf.freebuffer()
    frag["golden_md5"] = md5_of(bg)
    d = np.abs(bg - ref)
    frag["max_abs_err"] = "%.6e" % float(d.max())
    frag["worst_idx"]   = str(int(d.argmax()))
    frag["gate"] = "PASS" if np.array_equal(bg, ref) else "FAIL"
    return bg, frag

def run_stim_file(ip, dma, ibc, obc, x, out_buf):
    """Chunked state-carried run; reset on chunk 0 only. Certified semantics
    (verbatim from BOARDS2-REV4)."""
    rm = ip.register_map
    n_total = x.size
    t0 = time.time(); pos = 0; ci = 0
    while pos < n_total:
        n = min(CHUNK, n_total - pos)
        ibc[:n] = x[pos:pos+n]; ibc.flush()
        rm.mode = 1; rm.n_samples = n; rm.reset_state = 1 if ci == 0 else 0
        dma.recvchannel.transfer(obc[:n] if n < CHUNK else obc)
        start_kernel(ip)
        dma.sendchannel.transfer(ibc[:n] if n < CHUNK else ibc)
        dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
        obc.invalidate()
        out_buf[pos:pos+n] = obc[:n]
        pos += n; ci += 1
    dt = time.time() - t0
    return {"n_samples": str(n_total), "secs": "%.2f" % dt,
            "ksamp_s": "%.0f" % (n_total/dt/1e3),
            "out_md5": md5_of(out_buf[:n_total])}

FIELDS = ["timestamp","cell","width","bit","stim","stim_md5","status","gate",
          "max_abs_err","worst_idx","golden_md5","out_md5","out_file",
          "n_samples","secs","ksamp_s","fclk_mhz","note"]

def append_row(row):
    new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow(row); f.flush(); os.fsync(f.fileno())

def load_done_set():
    done = set()
    if os.path.exists(RESULTS):
        with open(RESULTS) as f:
            for r in csv.DictReader(f):
                if r.get("status") == "OK":
                    done.add((r["cell"], int(r["width"]), r["stim"],
                              r.get("stim_md5", "")))
    return done

def free_gb():
    st = os.statvfs(SESSION_DIR)
    return st.f_bavail*st.f_frsize/1e9

# ---- hard provenance gates ----
assert os.path.exists(GOLDEN_IN), "stage golden_input.bin first"
golden_x = np.fromfile(GOLDEN_IN, dtype=np.float32)
assert golden_x.size == GOLDEN_N
GOLDEN_FILE_MD5 = file_md5(GOLDEN_IN)
assert GOLDEN_FILE_MD5 == EXPECTED_MD5["golden_input.bin"], \
    "golden_input.bin MD5 mismatch: %s" % GOLDEN_FILE_MD5

lenny_man = {}
assert os.path.exists(STIM_MAN), "stage stim_manifest_session1.csv first"
with open(STIM_MAN) as f:
    for r in csv.DictReader(f):
        lenny_man[r["file"]] = r["md5"].lower()

STIM_MD5, STIM_N = {}, {}
for sname, fn in STIMS.items():
    p = os.path.join(SESSION_DIR, fn)
    assert os.path.exists(p), "MISSING stimulus: %s" % p
    m = file_md5(p)
    exp = EXPECTED_MD5.get(fn, lenny_man.get(fn))
    assert exp is not None, "no expected MD5 for %s (manifest incomplete)" % fn
    assert m == exp, "STIM MD5 MISMATCH %s: staged %s expected %s" % (fn, m, exp)
    STIM_MD5[sname] = m
    STIM_N[sname] = os.path.getsize(p) // 4
    print("stim %-8s %9d samples  MD5 %s  VERIFIED" % (sname, STIM_N[sname], m))
print("helpers defined; all stimulus provenance gates PASS")


## 3. Plan, run, summarize

In [ ]:
# J1. Build job list + preflight (safe to run any time)
jobs = []
with open(MANIFEST) as f:
    for r in csv.DictReader(f):
        if r["cell"] in STIM_CELLS:
            jobs.append({"cell": r["cell"], "width": int(r["width"]), "bit": r["bit"]})
jobs.sort(key=lambda e: (STIM_CELLS.index(e["cell"]), e["width"]))
done = load_done_set()
n_todo = sum(1 for e in jobs for s in STIMS
             if (e["cell"], e["width"], s, STIM_MD5[s]) not in done)
print("session 1: %d bitstreams x %d stims -> %d runs todo" % (len(jobs), len(STIMS), n_todo))
missing = []
for e in jobs:
    tag = "%s_w%d" % (e["cell"], e["width"])
    for p in (e["bit"], e["bit"].replace(".bit", ".hwh"), "ref_gold_%s.f32" % tag):
        if not os.path.exists(os.path.join(SESSION_DIR, p)):
            missing.append(p)
if missing:
    print("MISSING (stage before J2):")
    for p in missing: print("   ", p)
else:
    print("all %d bitstreams + hwh + B1 refs present" % len(jobs))
need_gb = sum(STIM_N[s] for s in STIMS) * 4 * len(jobs) / 1e9
print("outputs to write: ~%.1f GB | free now: %.1f GB (mid-session pull if short)"
      % (need_gb, free_gb()))


In [ ]:
# J2. THE SESSION LOOP -- start it and leave it alone (resumable)
done = load_done_set()
STIM_ORDER = list(MUSIC_STIMS) + list(BATT_STIMS)

for i, e in enumerate(jobs):
    pend = [s for s in STIM_ORDER if (e["cell"], e["width"], s, STIM_MD5[s]) not in done]
    tag = "%s_w%d" % (e["cell"], e["width"])
    gkey = (e["cell"], e["width"], "golden", GOLDEN_FILE_MD5)
    if not pend and gkey in done:
        print("[%s] %d/%d %-18s all done, skip" % (time.strftime("%H:%M"), i+1, len(jobs), tag))
        continue
    if free_gb() < 0.4:
        print("!! free space %.2f GB < 0.4 GB -- HARD STOP. Pull+verify on Lenny,"
              " delete pulled outputs (C1), re-run J2." % free_gb())
        break
    ibc = obc = None
    try:
        ol, ip, dma = load_bitstream(e["bit"])
        ref = np.fromfile(os.path.join(SESSION_DIR, "ref_gold_%s.f32" % tag),
                          dtype=np.float32)
        bg, gfrag = run_golden(ip, dma, golden_x, ref)
        bg.tofile(os.path.join(SESSION_DIR, "s1_gold_%s.f32" % tag))
        if gkey not in done:
            row = {k: "" for k in FIELDS}
            row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
                       width=e["width"], bit=e["bit"], stim="golden",
                       stim_md5=GOLDEN_FILE_MD5, status="OK",
                       fclk_mhz="%.1f" % Clocks.fclk0_mhz, **gfrag)
            append_row(row)
        if gfrag["gate"] != "PASS":
            print("[%s] %-18s GOLDEN GATE %s vs B1 ref -- stims SKIPPED"
                  % (time.strftime("%H:%M"), tag, gfrag["gate"]))
            continue
        ibc = allocate(shape=(CHUNK,), dtype=np.float32)
        obc = allocate(shape=(CHUNK,), dtype=np.float32)
        for sname in pend:
            row = {k: "" for k in FIELDS}
            row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
                       width=e["width"], bit=e["bit"], stim=sname,
                       stim_md5=STIM_MD5[sname], gate=gfrag["gate"],
                       golden_md5=gfrag["golden_md5"],
                       fclk_mhz="%.1f" % Clocks.fclk0_mhz)
            x = out_buf = None
            try:
                x = np.fromfile(os.path.join(SESSION_DIR, STIMS[sname]),
                                dtype=np.float32)          # lazy load (RAM budget)
                out_buf = np.empty(x.size, dtype=np.float32)
                frag = run_stim_file(ip, dma, ibc, obc, x, out_buf)
                if sname in MUSIC_STIMS:
                    oname = "board_%s_%s.f32" % (sname, tag)
                else:
                    oname = "s2_%s_%s.f32" % (sname, tag)   # battery naming (locked)
                out_buf.tofile(os.path.join(SESSION_DIR, oname))
                row.update(frag); row["out_file"] = oname; row["status"] = "OK"
            except TimeoutError as ex:
                row.update(status="TIMEOUT", note=str(ex))
            except Exception as ex:
                row.update(status="ERROR", note=type(ex).__name__ + ": " + str(ex)[:120])
            finally:
                del x, out_buf
            append_row(row)
            print("[%s] %d/%d %-18s %-8s %-7s %s k/s" % (time.strftime("%H:%M"),
                  i+1, len(jobs), tag, sname, row["status"], row.get("ksamp_s","-")))
    except Exception as ex:
        row = {k: "" for k in FIELDS}
        row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), cell=e["cell"],
                   width=e["width"], bit=e["bit"], stim="load", status="ERROR",
                   note=type(ex).__name__ + ": " + str(ex)[:120])
        append_row(row)
        print("[%s] %-18s LOAD ERROR: %s" % (time.strftime("%H:%M"), tag, ex))
    finally:
        for b in (ibc, obc):
            try:
                if b is not None: b.freebuffer()
            except Exception:
                pass
print("session #1 pass complete -- results in", RESULTS)


In [ ]:
# J3. Results summary (provenance-aware)
rows = list(csv.DictReader(open(RESULTS)))
ok = [r for r in rows if r["status"] == "OK"]
gates = [r for r in rows if r["stim"] == "golden"]
print("rows: %d | OK: %d | not OK: %d | golden gates: %d (%d PASS)"
      % (len(rows), len(ok), len(rows)-len(ok), len(gates),
         sum(1 for r in gates if r.get("gate") == "PASS")))
for r in rows:
    if r["status"] != "OK" or r.get("gate") == "FAIL":
        print("  !!", r["cell"], "w"+r["width"], r["stim"], r["status"], r.get("gate",""), r.get("note",""))
per = {}
for r in ok:
    if r["stim"] != "golden":
        per.setdefault((r["cell"], r["stim"]), []).append(int(r["width"]))
for k in sorted(per):
    print("  %-12s %-8s %d/17 widths" % (k[0], k[1], len(per[k])))


In [ ]:
# C1. Cleanup helper -- run ONLY after Lenny-side pull is hash-verified.
# Edit DELETE_TAGS to the configs whose outputs are verified on the NAS,
# e.g. DELETE_TAGS = ["rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w8", "gt_max_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w8", "gt_mod_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["fl_max_w10", "fl_max_w11", "fl_max_w12", "fl_max_w13", "fl_max_w14", "fl_max_w15", "fl_max_w16", "fl_max_w8", "fl_max_w9", "gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w21", "gt_mod_w22", "gt_mod_w23", "gt_mod_w24", "gt_mod_w8", "gt_mod_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["fl_max_w10", "fl_max_w11", "fl_max_w12", "fl_max_w13", "fl_max_w14", "fl_max_w15", "fl_max_w16", "fl_max_w17", "fl_max_w18", "fl_max_w19", "fl_max_w20", "fl_max_w21", "fl_max_w22", "fl_max_w23", "fl_max_w24", "fl_max_w8", "fl_max_w9", "gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w21", "gt_mod_w22", "gt_mod_w23", "gt_mod_w24", "gt_mod_w8", "gt_mod_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["fl_max_w10", "fl_max_w11", "fl_max_w12", "fl_max_w13", "fl_max_w14", "fl_max_w15", "fl_max_w16", "fl_max_w17", "fl_max_w18", "fl_max_w19", "fl_max_w20", "fl_max_w21", "fl_max_w22", "fl_max_w23", "fl_max_w24", "fl_max_w8", "fl_max_w9", "fl_mod_w10", "fl_mod_w11", "fl_mod_w12", "fl_mod_w13", "fl_mod_w14", "fl_mod_w15", "fl_mod_w16", "fl_mod_w17", "fl_mod_w18", "fl_mod_w19", "fl_mod_w20", "fl_mod_w8", "fl_mod_w9", "gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w21", "gt_mod_w22", "gt_mod_w23", "gt_mod_w24", "gt_mod_w8", "gt_mod_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
#DELETE_TAGS = ["fl_max_w10", "fl_max_w11", "fl_max_w12", "fl_max_w13", "fl_max_w14", "fl_max_w15", "fl_max_w16", "fl_max_w17", "fl_max_w18", "fl_max_w19", "fl_max_w20", "fl_max_w21", "fl_max_w22", "fl_max_w23", "fl_max_w24", "fl_max_w8", "fl_max_w9", "fl_mod_w10", "fl_mod_w11", "fl_mod_w12", "fl_mod_w13", "fl_mod_w14", "fl_mod_w15", "fl_mod_w16", "fl_mod_w17", "fl_mod_w18", "fl_mod_w19", "fl_mod_w20", "fl_mod_w21", "fl_mod_w22", "fl_mod_w23", "fl_mod_w24", "fl_mod_w8", "fl_mod_w9", "gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w21", "gt_mod_w22", "gt_mod_w23", "gt_mod_w24", "gt_mod_w8", "gt_mod_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]

#DELETE_TAGS = ["fl_max_w10", "fl_max_w11", "fl_max_w12", "fl_max_w13", "fl_max_w14", "fl_max_w15", "fl_max_w16", "fl_max_w17", "fl_max_w18", "fl_max_w19", "fl_max_w20", "fl_max_w21", "fl_max_w22", "fl_max_w23", "fl_max_w24", "fl_max_w8", "fl_max_w9", "fl_mod_w10", "fl_mod_w11", "fl_mod_w12", "fl_mod_w13", "fl_mod_w14", "fl_mod_w15", "fl_mod_w16", "fl_mod_w17", "fl_mod_w18", "fl_mod_w19", "fl_mod_w20", "fl_mod_w21", "fl_mod_w22", "fl_mod_w23", "fl_mod_w24", "fl_mod_w8", "fl_mod_w9", "gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w21", "gt_mod_w22", "gt_mod_w23", "gt_mod_w24", "gt_mod_w8", "gt_mod_w9", "rodent_max_w10", "rodent_max_w11", "rodent_max_w12", "rodent_max_w13", "rodent_max_w14", "rodent_max_w15", "rodent_max_w16", "rodent_max_w17", "rodent_max_w18", "rodent_max_w19", "rodent_max_w20", "rodent_max_w8", "rodent_max_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]
DELETE_TAGS = ["fl_max_w10", "fl_max_w11", "fl_max_w12", "fl_max_w13", "fl_max_w14", "fl_max_w15", "fl_max_w16", "fl_max_w17", "fl_max_w18", "fl_max_w19", "fl_max_w20", "fl_max_w21", "fl_max_w22", "fl_max_w23", "fl_max_w24", "fl_max_w8", "fl_max_w9", "fl_mod_w10", "fl_mod_w11", "fl_mod_w12", "fl_mod_w13", "fl_mod_w14", "fl_mod_w15", "fl_mod_w16", "fl_mod_w17", "fl_mod_w18", "fl_mod_w19", "fl_mod_w20", "fl_mod_w21", "fl_mod_w22", "fl_mod_w23", "fl_mod_w24", "fl_mod_w8", "fl_mod_w9", "gt_max_w10", "gt_max_w11", "gt_max_w12", "gt_max_w13", "gt_max_w14", "gt_max_w15", "gt_max_w16", "gt_max_w17", "gt_max_w18", "gt_max_w19", "gt_max_w20", "gt_max_w21", "gt_max_w22", "gt_max_w23", "gt_max_w24", "gt_max_w8", "gt_max_w9", "gt_mod_w10", "gt_mod_w11", "gt_mod_w12", "gt_mod_w13", "gt_mod_w14", "gt_mod_w15", "gt_mod_w16", "gt_mod_w17", "gt_mod_w18", "gt_mod_w19", "gt_mod_w20", "gt_mod_w21", "gt_mod_w22", "gt_mod_w23", "gt_mod_w24", "gt_mod_w8", "gt_mod_w9", "rodent_max_w10", "rodent_max_w11", "rodent_max_w12", "rodent_max_w13", "rodent_max_w14", "rodent_max_w15", "rodent_max_w16", "rodent_max_w17", "rodent_max_w18", "rodent_max_w19", "rodent_max_w20", "rodent_max_w21", "rodent_max_w22", "rodent_max_w23", "rodent_max_w24", "rodent_max_w8", "rodent_max_w9", "rodent_mod_w10", "rodent_mod_w11", "rodent_mod_w12", "rodent_mod_w13", "rodent_mod_w14", "rodent_mod_w15", "rodent_mod_w16", "rodent_mod_w17", "rodent_mod_w18", "rodent_mod_w19", "rodent_mod_w20", "rodent_mod_w21", "rodent_mod_w22", "rodent_mod_w23", "rodent_mod_w24", "rodent_mod_w8", "rodent_mod_w9"]

import glob
freed = 0
for tag in DELETE_TAGS:
    for pat in ("board_*_%s.f32" % tag, "s2_*_%s.f32" % tag):
        for p in glob.glob(os.path.join(SESSION_DIR, pat)):
            freed += os.path.getsize(p); os.remove(p); print("rm", os.path.basename(p))
print("freed %.2f GB | free now %.1f GB" % (freed/1e9, free_gb()))


## Close of session
1. On-board manifest: run in a board terminal —
   `cd ~/jupyter_notebooks/session1 && md5sum *.f32 *.csv > session1_md5.txt`
2. Lenny pull → `M:\hope\board_session1_2026-08-15\` → byte-verify vs
   `session1_md5.txt` + write-time `out_md5` (verify_s2_pull_REV2 pattern).
3. Rolling NMR + float-ref ESR + battery scoring start as cells verify.
4. RTNeural ARM baseline row at session close (fallback Mon 8/17).